In [13]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# Path to dataset
dataSet = r"C:\Users\rohan\CW2_Datasets\Dataset2\files"


ImageSize = (150, 150)
BatchSize = 32
epochs = 12

HeartLabels = [
    "atherosclerosis_of_the_aorta",
    "cardiomegaly",
    "pneumonia",
    "pneumosclerosis",
    "post_inflammatory_changes",
    "post_traumatic_ribs_deformation"
]

# Normalisation and  validation 
generator = ImageDataGenerator(rescale=1.0 / 255, validation_split=0.2)

train_loader = generator.flow_from_directory(
    dataSet,
    target_size=ImageSize,
    batch_size=BatchSize,
    class_mode="categorical",
    subset="training",
    shuffle=True
)

ValidationLoader = generator.flow_from_directory(
    dataSet,
    target_size=ImageSize,
    batch_size=BatchSize,
    class_mode="categorical",
    subset="validation",
    shuffle=False
)

# The images come in 17 disease classes, but the task asks for a binary decision:
# heart-related or not. So here I convert each folder label into 0/1 based on whether
# the disease is in the HeartLabels list.
def label_creator(source_gen):
    class_map = list(source_gen.class_indices.keys())

    while True:
        images, labels = next(source_gen)         
        label_ids = np.argmax(labels, axis=1)     

        binary_targets = np.array(
            [1 if class_map[i] in HeartLabels else 0 for i in label_ids],
            dtype=np.int32
        )
        yield images, binary_targets

TrainingData = label_creator(train_loader)
ValidationData = label_creator(ValidationLoader)


model = models.Sequential([
    tf.keras.Input(shape=(ImageSize[0], ImageSize[1], 3)),

    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D(pool_size=(2, 2)),

    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D(pool_size=(2, 2)),

    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.fit(
    TrainingData,
    steps_per_epoch=len(train_loader),
    validation_data=ValidationData,
    validation_steps=len(ValidationLoader),
    epochs=epochs
)


TrueLabels = []
PredictedLabels = []

for i in range(len(ValidationLoader)):
    images, labels = next(ValidationData)
    probs = model.predict(images, verbose=0).reshape(-1)
    preds = (probs >= 0.5).astype(int)

    TrueLabels.extend(labels.tolist())
    PredictedLabels.extend(preds.tolist())

TrueLabels = np.array(TrueLabels, dtype=np.int32)
PredictedLabels = np.array(PredictedLabels, dtype=np.int32)


print("Validation Accuracy:", accuracy_score(TrueLabels, PredictedLabels))
print("\nConfusion Matrix:")
print(confusion_matrix(TrueLabels, PredictedLabels, labels=[0, 1]))

print("\nClassification Report:")
print(classification_report(
    TrueLabels, PredictedLabels,
    labels=[0, 1],
    target_names=["Non-heart-related", "Heart-related"],
    zero_division=0
))


Found 84 images belonging to 17 classes.
Found 13 images belonging to 17 classes.
Epoch 1/12
3/3 ━━━━━━━━━━━━━━━━━━━━ 7s 2s/step - accuracy: 0.5476 - loss: 2.4040 - val_accuracy: 0.3846 - val_loss: 0.9270
Epoch 2/12
3/3 ━━━━━━━━━━━━━━━━━━━━ 6s 3s/step - accuracy: 0.3571 - loss: 0.8291 - val_accuracy: 0.3846 - val_loss: 0.7558
Epoch 3/12
3/3 ━━━━━━━━━━━━━━━━━━━━ 9s 3s/step - accuracy: 0.5238 - loss: 0.7000 - val_accuracy: 0.6154 - val_loss: 0.6375
Epoch 4/12
3/3 ━━━━━━━━━━━━━━━━━━━━ 8s 2s/step - accuracy: 0.6429 - loss: 0.6500 - val_accuracy: 0.6154 - val_loss: 0.6279
Epoch 5/12
3/3 ━━━━━━━━━━━━━━━━━━━━ 8s 2s/step - accuracy: 0.6429 - loss: 0.6529 - val_accuracy: 0.6154 - val_loss: 0.6229
Epoch 6/12
3/3 ━━━━━━━━━━━━━━━━━━━━ 8s 3s/step - accuracy: 0.6429 - loss: 0.6334 - val_accuracy: 0.6154 - val_loss: 0.6258
Epoch 7/12
3/3 ━━━━━━━━━━━━━━━━━━━━ 8s 3s/step - accuracy: 0.6429 - loss: 0.6222 - val_accuracy: 0.6154 - val_loss: 0.6300
Epoch 8/12
3/3 ━━━━━━━━━━━━━━━━━━━━ 8s 3s/step - accuracy